# Statistics for Data Analysts/Engineers — Module 6: Confidence Intervals

In Module 5 we learned that the sample mean has its own standard error and (thanks to the
CLT) an approximately normal distribution. Now we'll use that to build a **confidence
interval** -- instead of reporting a single number as our "best guess" of the true
population mean, we'll report an entire range of values, along with how much we trust it.

## Table of contents
1. [What a confidence interval is (and isn't)](#sec1)
2. [Confidence interval for a mean — unknown sigma (the t-distribution)](#sec2)
3. [How confidence level and sample size affect interval width](#sec3)
4. [Confidence interval for a proportion](#sec4)
5. [Verification: simulating confidence-interval coverage](#sec5)
6. [Summary and exercises](#sec6)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. What a confidence interval is (and isn't)

A **confidence interval (CI)** at, say, the 95% level is a range of values computed from
a sample such that, if you repeated the entire sampling-and-computing procedure a very
large number of times, **95% of the intervals built this way would contain the true,
unknown population parameter**.

This is a subtle but important distinction:
- ❌ WRONG interpretation: "there's a 95% chance the true mean lies in THIS specific
  interval".
- ✅ CORRECT interpretation: "this METHOD of building intervals hits the true mean 95% of
  the time (in the long run)".

The true population mean either is or isn't in our specific interval -- there's no
randomness left there. What's random is the METHOD, not any particular result. We'll see
this very literally in the simulation in section 5.

> ⚠️ **This is the most commonly misunderstood concept in all of statistics**
>
> Even experienced analysts sometimes say "there's a 95% chance the truth is in this interval" as a mental shorthand -- that's informally tolerated in casual conversation, but formally incorrect, and it's worth understanding the difference, especially when explaining results to someone who will act on them.

<a id="sec2"></a>
## 2. Confidence interval for a mean — unknown sigma (the t-distribution)

The general formula for a confidence interval of a mean:

```
x-bar +/- (critical value) * SE
```

In Module 5 we computed `SE = sigma / sqrt(n)`, assuming we knew `sigma` (the
**population's** standard deviation). In practice we almost never know `sigma` -- we
estimate it from the sample (`s`, the sample standard deviation). When `sigma` is
unknown, instead of a critical value from the normal distribution (z) we use a critical
value from the **Student's t-distribution**, which has "fatter tails" (is more cautious),
especially for small samples. The t-distribution has a **degrees of freedom** parameter
(`df = n - 1`) -- the more degrees of freedom, the more the t-distribution resembles the
normal distribution.

In [ ]:
import numpy as np
from scipy import stats

sample = customers["monthly_spend"].sample(n=40, random_state=1)

x_bar = sample.mean()
s = sample.std(ddof=1)
n = len(sample)
se = s / np.sqrt(n)
df = n - 1

confidence_level = 0.95
t_critical = stats.t.ppf((1 + confidence_level) / 2, df=df)

lower_bound = x_bar - t_critical * se
upper_bound = x_bar + t_critical * se

print(f"Sample mean: {x_bar:.2f}")
print(f"t critical value (df={df}): {t_critical:.3f}")
print(f"95% CI: [{lower_bound:.2f}, {upper_bound:.2f}]")


> 📊 **scipy.stats.t.interval does this in one line**
>
> The manual computation above shows what's happening under the hood, but in practice it's more convenient to use a ready-made function: `stats.t.interval(confidence, df, loc=x_bar, scale=se)` returns exactly the same pair of bounds.

In [ ]:
ready_ci = stats.t.interval(confidence_level, df=df, loc=x_bar, scale=se)
print(ready_ci)


> ⚠️ **t or z? A practical rule**
>
> Formally, the t-distribution is correct whenever the population's `sigma` is unknown (almost always in practice) -- even for large `n`. For large samples (well above `n=30-50`), the t-distribution ends up resembling the normal distribution anyway, so the difference in the result is negligible, but the rule "unknown sigma -> t" is a safe default regardless of sample size.

<a id="sec3"></a>
## 3. How confidence level and sample size affect interval width

Two factors affect the width of a confidence interval:

- **A higher confidence level** (e.g. 99% instead of 95%) -> a **wider** interval (to be
  "more certain", you need to give yourself a bigger margin).
- **A larger sample** (`n`) -> a **narrower** interval (smaller standard error, exactly
  like in Module 5).

In [ ]:
import matplotlib.pyplot as plt

levels = [0.80, 0.90, 0.95, 0.99]
widths = []
for level in levels:
    lower, upper = stats.t.interval(level, df=df, loc=x_bar, scale=se)
    widths.append(upper - lower)

plt.figure(figsize=(7, 4))
plt.bar([f"{int(p*100)}%" for p in levels], widths)
plt.xlabel("Confidence level")
plt.ylabel("Interval width")
plt.title("Higher confidence level = wider interval (same n)")
plt.show()


In [ ]:
sample_sizes = [10, 20, 40, 80, 160]
widths_by_n = []
for sn in sample_sizes:
    sample_n = customers["monthly_spend"].sample(n=sn, random_state=1)
    s_n = sample_n.std(ddof=1)
    se_n = s_n / np.sqrt(sn)
    lower, upper = stats.t.interval(0.95, df=sn - 1, loc=sample_n.mean(), scale=se_n)
    widths_by_n.append(upper - lower)

plt.figure(figsize=(7, 4))
plt.plot(sample_sizes, widths_by_n, marker="o")
plt.xlabel("Sample size (n)")
plt.ylabel("95% confidence interval width")
plt.title("Larger sample = narrower confidence interval")
plt.show()


<a id="sec4"></a>
## 4. Confidence interval for a proportion

When we care not about the mean of a continuous number but about a **proportion/rate**
(e.g. what fraction of customers convert), we use a slightly different formula -- based
on the normal approximation of the binomial distribution (Module 4):

```
p-hat +/- z * sqrt(p-hat * (1 - p-hat) / n)
```

where `p-hat` is the observed proportion in the sample and `z` is the critical value from
the normal distribution (not t -- for proportions, z is the standard choice even with an
"unknown sigma", because the variance of a proportion `p*(1-p)` depends only on `p`
itself).

In [ ]:
p_hat = customers["converted"].mean()
n = len(customers)
z_critical = stats.norm.ppf(0.975)  # 95% CI -> 2.5% in each tail

se_proportion = np.sqrt(p_hat * (1 - p_hat) / n)
lower_p = p_hat - z_critical * se_proportion
upper_p = p_hat + z_critical * se_proportion

print(f"Observed conversion rate: {p_hat:.3f}")
print(f"95% CI: [{lower_p:.3f}, {upper_p:.3f}]")


`statsmodels` has a ready-made function for the same thing (`proportion_confint`), worth
knowing since it also automatically handles variants for small/extreme proportions:

In [ ]:
from statsmodels.stats.proportion import proportion_confint

num_conversions = customers["converted"].sum()
lower_sm, upper_sm = proportion_confint(num_conversions, n, alpha=0.05, method="normal")
print(f"95% CI (statsmodels): [{lower_sm:.3f}, {upper_sm:.3f}]")


<a id="sec5"></a>
## 5. Verification: simulating confidence-interval coverage

Let's go back to the definition from section 1 and verify it literally: let's build 200
different 95% confidence intervals from 200 different samples and count how many of them
actually contain the true population mean (which, just like in Module 5, we exceptionally
know here).

In [ ]:
rng = np.random.default_rng(7)
population_mu = customers["monthly_spend"].mean()

num_simulations = 200
contains_truth = 0
intervals = []

for _ in range(num_simulations):
    sim_sample = rng.choice(customers["monthly_spend"], size=40, replace=False)
    sim_x_bar = sim_sample.mean()
    sim_se = sim_sample.std(ddof=1) / np.sqrt(40)
    sim_lower, sim_upper = stats.t.interval(0.95, df=39, loc=sim_x_bar, scale=sim_se)
    intervals.append((sim_lower, sim_upper))
    if sim_lower <= population_mu <= sim_upper:
        contains_truth += 1

print(f"Share of intervals containing the true mean: {contains_truth / num_simulations:.3f}")
print("Expected (in the long run): about 0.95")


In [ ]:
plt.figure(figsize=(9, 6))
for i, (lower_i, upper_i) in enumerate(intervals[:50]):
    color = "steelblue" if lower_i <= population_mu <= upper_i else "crimson"
    plt.plot([lower_i, upper_i], [i, i], color=color)
plt.axvline(population_mu, color="black", linestyle="--", label="True population mean")
plt.xlabel("Monthly spend")
plt.ylabel("Simulation number (first 50)")
plt.title("Red intervals did NOT capture the true mean")
plt.legend()
plt.show()


> 📊 **This is the one visualization worth remembering**
>
> This chart is maybe the single most telling picture in this whole module: the vast majority of intervals (blue) do contain the true, horizontal line, but a handful (red) -- despite the method being applied correctly -- don't. That's exactly what "95% confidence" means: not that EVERY interval is good, but that the method is wrong on average about once every 20 times.

<a id="sec6"></a>
## 6. Summary and exercises

In this module we covered:
- the correct (and incorrect) interpretation of a confidence interval,
- building a CI for a mean using the t-distribution (when the population's `sigma` is
  unknown),
- how confidence level and sample size affect interval width,
- a CI for a proportion, based on the normal approximation,
- a simulation confirming the literal meaning of "confidence level" as the method's
  long-run hit rate, not the probability of a single interval.

> 📝 **Exercise 1: CI for age**
>
> Draw a sample of 50 customers (`random_state=2`) and compute a 95% confidence interval for the mean age, using the t-distribution (by hand or `stats.t.interval`).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
age_sample = customers["age"].sample(n=50, random_state=2)
x_bar_a = age_sample.mean()
se_a = age_sample.std(ddof=1) / np.sqrt(50)
print(stats.t.interval(0.95, df=49, loc=x_bar_a, scale=se_a))
```
</details>

> 📝 **Exercise 2: 99% CI for spend**
>
> For the same sample from section 2 (`sample`, n=40), compute a confidence interval at the 99% level instead of 95%. How much wider is it compared to the interval from section 2?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
ci_99 = stats.t.interval(0.99, df=df, loc=x_bar, scale=se)
width_99 = ci_99[1] - ci_99[0]
width_95 = upper_bound - lower_bound
print(f"CI 99%: {ci_99}")
print(f"Width 99%: {width_99:.2f} vs width 95%: {width_95:.2f}")
```
</details>

> 📝 **Exercise 3: CI for the proportion by group**
>
> Compute a separate 95% confidence interval for the conversion rate in group A and in group B (use `proportion_confint` or the manual formula from section 4). Do the intervals overlap, or are they disjoint?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
from statsmodels.stats.proportion import proportion_confint

for group in ["A", "B"]:
    group_data = customers[customers["campaign_group"] == group]
    num_conv = group_data["converted"].sum()
    n_group = len(group_data)
    lower_g, upper_g = proportion_confint(num_conv, n_group, alpha=0.05, method="normal")
    print(f"Group {group}: p_hat={num_conv / n_group:.3f}, CI=[{lower_g:.3f}, {upper_g:.3f}]")
```

If the intervals do NOT overlap, that's a strong informal hint that the difference between groups is real -- but a formal test (Module 9, chi-square) is a more precise tool for this question than just comparing intervals.
</details>

> 🔥 **Challenge: your own CI function**
>
> Write a function `confidence_interval(data, level=0.95)` that takes a numeric Series and returns a tuple `(lower, upper)` for a t-distribution-based confidence interval of the mean. Test it on the `monthly_spend`, `age`, and `satisfaction` columns (on the full `customers` dataset, n=400).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
def confidence_interval(data, level=0.95):
    x_bar_f = data.mean()
    se_f = data.std(ddof=1) / np.sqrt(len(data))
    df_f = len(data) - 1
    return stats.t.interval(level, df=df_f, loc=x_bar_f, scale=se_f)


for column in ["monthly_spend", "age", "satisfaction"]:
    print(column, confidence_interval(customers[column]))
```
</details>

## What's next?

In **Module 7** we'll move on to formal **hypothesis testing** -- learning to precisely
answer questions like "is this difference real, or just chance?", which we've only
gestured at informally so far (e.g. by comparing confidence intervals in exercise 3).
We'll cover the null hypothesis, p-values, and Type I/II errors -- the foundation for
every specific test in Modules 8-10.